In [33]:
import chromadb

In [34]:
from sentence_transformers import SentenceTransformer

In [35]:
from transformers import pipeline

In [36]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3481.23it/s]


In [37]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
model.eval()

def generator(text, **kwargs):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    generation_options = {
        "max_new_tokens": 128,
    }
    generation_options.update(kwargs)

    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_options)

    return [{"generated_text": tokenizer.decode(outputs[0], skip_special_tokens=True)}]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1177.60it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [38]:
client = chromadb.PersistentClient(
    path="d:/itw/chromadb"
)

In [39]:
collection = client.get_or_create_collection(
    name="itw10",
    metadata={"hnsw:space": "cosine"},)


documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A journey of a thousand miles begins with a single step.",
    "To be or not to be, that is the question.",
    "All that glitters is not gold.",
    "I think, therefore I am.",
    "The only thing we have to fear is fear itself.",
    "In the beginning, God created the heavens and the earth.",
]

document_ids = [f"doc_{i}" for i in range(len(documents))]
metadata = [{"source": f"source_{i}"} for i in range(len(documents))]



In [40]:
document_embeddings = embedding_model.encode(documents)

In [41]:
collection.upsert(
    documents=documents,
    embeddings=document_embeddings.tolist(),
    metadatas=metadata,
    ids=document_ids,
)

In [42]:
def retrieve_documents(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
    )
    return (
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    )

In [43]:
def create_prompt(query, context):
    context = "\n".join(context)
    prompt = f"""
    answer the question based on the context below, and if the question can't be answered based on the context, say "I don't know"
    
    context: {context}
    question: {query}
    answer:
    """
    return prompt

In [44]:
def rag_chatbot(query):
    retrieved_documents, retrieved_metadata, distances = (
        retrieve_documents(query, top_k=3)
    )
    prompt = create_prompt(query, retrieved_documents)
    result = generator(prompt,
                       max_new_tokens=150,
                       do_sample=False)

    answer = result[0]["generated_text"]

    return (answer,
            retrieved_documents,
            distances,
            retrieved_metadata,)

In [45]:
answer, retrieved_documents, distances, retrieved_metadata = rag_chatbot(
    "What is the only thing we have to fear?"
)

print("Answer:", answer)
for doc, dist, meta in zip(retrieved_documents, distances, retrieved_metadata):
    print(f"{dist:.4f}  {meta['source']}  {doc}")

Answer: fear itself
0.1046  source_5  The only thing we have to fear is fear itself.
0.7044  source_2  To be or not to be, that is the question.
0.8591  source_6  In the beginning, God created the heavens and the earth.


In [46]:
while True:
    user_question = input("Enter your question (or type 'exit' to quit): ")
    if not user_question.strip():
        print("Please enter a valid question")
        continue
    if user_question.strip().lower() == 'exit':
        print("Goodbye!")
        break

    answer, retrieved_documents, distances, retrieved_metadata = rag_chatbot(user_question)
    print("\n--- RAG Response ---")
    print(answer)
    print("\nSources:")
    for doc, dist, meta in zip(retrieved_documents, distances, retrieved_metadata):
        print(f"  {dist:.4f}  {meta['source']}  {doc}")
    print("---------------------\n")


--- RAG Response ---
I don't know

Sources:
  0.7734  source_0  The quick brown fox jumps over the lazy dog.
  0.8139  source_4  I think, therefore I am.
  0.8800  source_2  To be or not to be, that is the question.
---------------------


--- RAG Response ---
The quick brown fox

Sources:
  0.4400  source_0  The quick brown fox jumps over the lazy dog.
  0.9137  source_6  In the beginning, God created the heavens and the earth.
  0.9256  source_1  A journey of a thousand miles begins with a single step.
---------------------

Goodbye!
